# CatBoost SHAP Feature Importance — Leakage-Safe Patient-Grouped CV

This notebook follows the modelling approach in **modeling-daily.ipynb**:
- **Model**: `CatBoostRegressor(iterations=1000, lr=0.1, depth=3, loss=RMSE)`
- **Preprocessing**: `StandardScaler → MinMaxScaler` (fit on training partition only)
- **Input rows**: raw daily sensor rows from `maison-llf-features.csv` (1 008 rows)
- **Targets**: SIS total, OHS total, OKS total

**Key departure from modeling-daily**: SHAP is computed on the **training partition only** (not the test set) to prevent data leakage. Patient-grouped 5-fold CV ensures no patient bleeds between train and test splits.

Two visualisations are produced for each target:
1. **Histogram / bar chart** — fold-averaged mean |SHAP| (one bar per feature)
2. **Beeswarm plot** — raw per-sample SHAP values pooled across all training folds

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder

SEED    = 42
N_FOLDS = 5
DATA_DIR = Path.cwd()

random.seed(SEED)
np.random.seed(SEED)
plt.style.use('ggplot')

SHAP_TARGETS = ['sis', 'ohs', 'oks']
_pal = {'sis': 'steelblue', 'ohs': 'darkorange', 'oks': 'seagreen'}

# CatBoost hyperparameters — identical to modeling-daily.ipynb
_cb_kw = dict(
    iterations=1000, learning_rate=0.1, depth=3,
    loss_function='RMSE', verbose=False, random_seed=SEED,
)

## Data Loading and Feature Definitions

Feature columns are selected using the same modality prefixes as **modeling-daily.ipynb** (`acceleration`, `heartrate`, `motion`, `position`, `sleep`, `step`).  
`tug` and `chairstand` are excluded because they are concurrent clinical measures.

In [ ]:
raw = pd.read_csv(
    DATA_DIR / 'data' / 'maison-llf-features.csv',
    parse_dates=['timestamp', 'clinical-timestamp'],
)
raw = raw.sort_values(['participant', 'timestamp', 'clinical-timestamp']).reset_index(drop=True)

# Exclude clinical / target columns (same logic as temporal forecasting notebook)
sis_items    = [f'sis-{i:02d}' for i in range(1, 7)]
ohs_items    = [f'ohs-{i:02d}' for i in range(1, 13)]
oks_items    = [f'oks-{i:02d}' for i in range(1, 13)]
clinical_cols = sis_items + ohs_items + oks_items + ['sis', 'ohs', 'oks', 'tug', 'chairstand']
id_cols       = ['participant', 'timestamp', 'clinical-timestamp']
sensor_cols   = [c for c in raw.columns if c not in id_cols + clinical_cols]

print(f'rows={len(raw)}, patients={raw["participant"].nunique()}, sensor_cols={len(sensor_cols)}')
print(sensor_cols)

## Patient-Grouped 5-Fold CV

Patients are randomly assigned to folds using the same seed as the temporal forecasting notebook. Every daily row for a given patient stays entirely within one split (train/val/test), preventing patient-level leakage.

In [ ]:
all_pts = np.array(sorted(raw['participant'].unique()))
rng     = np.random.default_rng(SEED)
shuffled = all_pts.copy()
rng.shuffle(shuffled)
test_folds = [np.array(f, dtype=int) for f in np.array_split(shuffled, N_FOLDS)]

def idx_for_patients(pts):
    return np.where(raw['participant'].isin(pts))[0]

cv_folds_raw = []
for fi, tp in enumerate(test_folds, 1):
    vp      = test_folds[fi % N_FOLDS]
    train_p = np.array([p for p in all_pts if p not in set(tp) | set(vp)], dtype=int)
    cv_folds_raw.append({
        'fold':      fi,
        'train_ids': idx_for_patients(train_p),
        'val_ids':   idx_for_patients(vp),
        'test_ids':  idx_for_patients(tp),
    })

fold_table = pd.DataFrame([
    {'fold': f['fold'],
     'train_rows': len(f['train_ids']),
     'val_rows':   len(f['val_ids']),
     'test_rows':  len(f['test_ids'])}
    for f in cv_folds_raw
])
display(fold_table)

## Sensor SHAP — CatBoost (Training Folds Only)

For each fold the pipeline is:
1. Median-impute NaN values using training-fold statistics only
2. `StandardScaler` → `MinMaxScaler` (fit on training rows, applied to training rows)
3. Train `CatBoostRegressor` with early stopping evaluated on training loss
4. `shap.TreeExplainer` on training rows → raw per-sample SHAP values stored

**Histogram ranking**: fold-averaged mean |SHAP| (each fold weighted equally)  
**Beeswarm**: raw per-sample SHAP values from all training folds concatenated (3 024 rows)

In [ ]:
fold_shap_sensor = {t: [] for t in SHAP_TARGETS}
sensor_shap_raw  = {t: {'shap_values': [], 'X': []} for t in SHAP_TARGETS}

for fold_info in cv_folds_raw:
    t_ids = fold_info['train_ids']
    train = raw.iloc[t_ids]

    med = train[sensor_cols].median()
    Xr  = train[sensor_cols].fillna(med).values.astype(np.float32)

    scaler = StandardScaler()
    norm   = MinMaxScaler()
    X = norm.fit_transform(scaler.fit_transform(Xr))

    for target in SHAP_TARGETS:
        y_train = train[target].values
        m = CatBoostRegressor(**_cb_kw)
        m.fit(X, y_train, eval_set=(X, y_train),
              use_best_model=True, early_stopping_rounds=100)
        exp = shap.TreeExplainer(m)
        sv  = exp.shap_values(X)
        fold_shap_sensor[target].append(np.abs(sv).mean(axis=0))  # for histogram ranking
        sensor_shap_raw[target]['shap_values'].append(sv)           # raw, for beeswarm
        sensor_shap_raw[target]['X'].append(X)                      # matching feature matrix

    print(f'fold {fold_info["fold"]} done  (train={len(t_ids)} rows)')

# Concatenate raw arrays across folds
for target in SHAP_TARGETS:
    sensor_shap_raw[target]['shap_values'] = np.vstack(sensor_shap_raw[target]['shap_values'])
    sensor_shap_raw[target]['X']           = np.vstack(sensor_shap_raw[target]['X'])
    n = sensor_shap_raw[target]['shap_values'].shape[0]
    print(f'{target.upper()} pooled: {n} sample-fold rows')

# Build ranked DataFrames and save CSVs
sensor_shap_dfs = {}
for target in SHAP_TARGETS:
    avg = np.mean(fold_shap_sensor[target], axis=0)
    df  = (
        pd.DataFrame({'feature': sensor_cols, 'weights': avg})
        .sort_values('weights', ascending=False)
        .reset_index(drop=True)
    )
    sensor_shap_dfs[target] = df
    df.to_csv(DATA_DIR / f'shap_catboost_{target}_feature_importance.csv', index=False)
    print(f'{target.upper()} top-5: {df["feature"].head(5).tolist()}')

print('\nSensor SHAP CSVs saved.')

### Sensor Feature Importance — Histogram (Top 20)

In [ ]:
for target in SHAP_TARGETS:
    top20 = sensor_shap_dfs[target].head(20)
    fig, ax = plt.subplots(figsize=(9, 7))
    y_pos = np.arange(len(top20))
    ax.barh(y_pos, top20['weights'].values[::-1],
            color=_pal[target], edgecolor='white', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top20['feature'].values[::-1], fontsize=9)
    ax.set_xlabel('Mean |SHAP value| (fold-averaged, training data only)')
    ax.set_title(
        f'CatBoost – Top 20 Sensor Features → {target.upper()} Score\n'
        f'(histogram: fold-averaged mean |SHAP|)'
    )
    plt.tight_layout()
    plt.savefig(DATA_DIR / f'shap_catboost_sensor_{target}_top20.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: shap_catboost_sensor_{target}_top20.png')

### Sensor Feature Importance — Beeswarm (Top 20)

In [ ]:
for target in SHAP_TARGETS:
    top20_feat = sensor_shap_dfs[target]['feature'].head(20).tolist()
    top20_idx  = [sensor_cols.index(f) for f in top20_feat]
    sv_pool    = sensor_shap_raw[target]['shap_values'][:, top20_idx]
    X_pool     = sensor_shap_raw[target]['X'][:, top20_idx]
    n_rows     = sv_pool.shape[0]

    shap.summary_plot(sv_pool, X_pool, feature_names=top20_feat,
                      max_display=20, show=False)
    fig = plt.gcf()
    fig.set_size_inches(10, 8)
    fig.suptitle(
        f'CatBoost SHAP Beeswarm – Top 20 Sensor Features → {target.upper()} Score\n'
        f'(pooled training-fold SHAP, {n_rows} sample-fold rows across 5 folds)',
        fontsize=11, y=1.02,
    )
    plt.savefig(DATA_DIR / f'shap_catboost_sensor_{target}_beeswarm.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: shap_catboost_sensor_{target}_beeswarm.png')

## Demographic SHAP — CatBoost (Training Folds Only)

Categorical demographic features (sex, fracture-type, relationship, education, work, ethnicity) are ordinal-encoded before the `StandardScaler → MinMaxScaler` pipeline. The same 5 patient-grouped folds are used.

In [ ]:
demo_raw_df   = pd.read_csv(DATA_DIR / 'data' / 'maison-llf-demographics.csv')
demo_feat_cols = ['sex', 'age', 'fracture-type', 'relationship', 'education', 'work', 'ethnicity']
_cat_cols      = [c for c in demo_feat_cols if c != 'age']

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
demo_encoded = demo_raw_df.copy()
demo_encoded[_cat_cols] = enc.fit_transform(demo_raw_df[_cat_cols].astype(str))

# Join demographics to every daily row (static per patient)
demo_daily = raw[['participant', 'sis', 'ohs', 'oks']].copy()
demo_daily = demo_daily.merge(
    demo_encoded[['participant'] + demo_feat_cols],
    on='participant', how='left',
)

fold_shap_demo = {t: [] for t in SHAP_TARGETS}
demo_shap_raw  = {t: {'shap_values': [], 'X': []} for t in SHAP_TARGETS}

for fold_info in cv_folds_raw:
    t_ids = fold_info['train_ids']
    train = demo_daily.iloc[t_ids]

    med = train[demo_feat_cols].median()
    Xr  = train[demo_feat_cols].fillna(med).values.astype(np.float32)

    scaler = StandardScaler()
    norm   = MinMaxScaler()
    X = norm.fit_transform(scaler.fit_transform(Xr))

    for target in SHAP_TARGETS:
        m = CatBoostRegressor(**_cb_kw)
        m.fit(X, train[target].values,
              eval_set=(X, train[target].values),
              use_best_model=True, early_stopping_rounds=100)
        exp = shap.TreeExplainer(m)
        sv  = exp.shap_values(X)
        fold_shap_demo[target].append(np.abs(sv).mean(axis=0))
        demo_shap_raw[target]['shap_values'].append(sv)
        demo_shap_raw[target]['X'].append(X)

    print(f'fold {fold_info["fold"]} done')

for target in SHAP_TARGETS:
    demo_shap_raw[target]['shap_values'] = np.vstack(demo_shap_raw[target]['shap_values'])
    demo_shap_raw[target]['X']           = np.vstack(demo_shap_raw[target]['X'])

demo_shap_dfs = {}
for target in SHAP_TARGETS:
    avg = np.mean(fold_shap_demo[target], axis=0)
    df  = (
        pd.DataFrame({'feature': demo_feat_cols, 'weights': avg})
        .sort_values('weights', ascending=False)
        .reset_index(drop=True)
    )
    demo_shap_dfs[target] = df
    df.to_csv(DATA_DIR / f'shap_catboost_demo_{target}_feature_importance.csv', index=False)
    print(f'\n{target.upper()} demographic SHAP:')
    print(df.to_string(index=False))

print('\nDemographic SHAP CSVs saved.')

### Demographic Feature Importance — Individual Histograms + Side-by-Side

In [ ]:
# Individual bar charts
for target in SHAP_TARGETS:
    df = demo_shap_dfs[target]
    fig, ax = plt.subplots(figsize=(8, 5))
    y_pos = np.arange(len(df))
    ax.barh(y_pos, df['weights'].values[::-1],
            color=_pal[target], edgecolor='white', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df['feature'].values[::-1], fontsize=10)
    ax.set_xlabel('Mean |SHAP value| (fold-averaged, training data only)')
    ax.set_title(f'CatBoost – Demographic Feature Importance → {target.upper()} Score')
    plt.tight_layout()
    plt.savefig(DATA_DIR / f'shap_catboost_demo_{target}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: shap_catboost_demo_{target}.png')

# Side-by-side
fig, axes = plt.subplots(1, 3, figsize=(19, 5), sharey=False)
for ax, target in zip(axes, SHAP_TARGETS):
    df = demo_shap_dfs[target]
    y_pos = np.arange(len(df))
    ax.barh(y_pos, df['weights'].values[::-1],
            color=_pal[target], edgecolor='white', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df['feature'].values[::-1], fontsize=10)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'→ {target.upper()} Score')
fig.suptitle(
    'CatBoost – Demographic Feature Importance (fold-averaged SHAP, training folds only)',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig(DATA_DIR / 'shap_catboost_demo_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_catboost_demo_combined.png')

### Demographic Feature Importance — Beeswarm

In [ ]:
for target in SHAP_TARGETS:
    sv_pool = demo_shap_raw[target]['shap_values']
    X_pool  = demo_shap_raw[target]['X']
    n_rows  = sv_pool.shape[0]

    shap.summary_plot(sv_pool, X_pool, feature_names=demo_feat_cols,
                      max_display=len(demo_feat_cols), show=False)
    fig = plt.gcf()
    fig.set_size_inches(9, 6)
    fig.suptitle(
        f'CatBoost SHAP Beeswarm – Demographic Features → {target.upper()} Score\n'
        f'(pooled training-fold SHAP, {n_rows} sample-fold rows across 5 folds)',
        fontsize=11, y=1.02,
    )
    plt.savefig(DATA_DIR / f'shap_catboost_demo_{target}_beeswarm.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: shap_catboost_demo_{target}_beeswarm.png')

## Why Histogram and Beeswarm Can Show Slightly Different Top Features

Both plots rank features by mean |SHAP value|, but they use **different averaging schemes** that can produce slightly different orderings for features with similar importance.

### Histogram (bar chart)
Ranking is computed as the **fold-averaged** mean |SHAP|:

$$\text{score}_j = \frac{1}{K} \sum_{k=1}^{K} \left( \frac{1}{n_k} \sum_{i \in \text{fold}_k} |\phi_{ij}^{(k)}| \right)$$

Each fold contributes **equally** to the average, regardless of how many training rows it contains ($n_k$ varies from 560 to 672 rows across the 5 folds here).

### Beeswarm plot
`shap.summary_plot` orders features by the mean |SHAP| of the **pooled** (concatenated) data:

$$\text{score}_j = \frac{1}{\sum_k n_k} \sum_{k=1}^{K} \sum_{i \in \text{fold}_k} |\phi_{ij}^{(k)}|$$

Here larger folds ($n_k = 672$) contribute **proportionally more** than smaller ones ($n_k = 560$).

### Why rankings can differ
For a feature whose SHAP importance is **inconsistent across folds** — high in the larger folds but low in the smaller ones — the pooled average will rank it higher than the fold-average. The reverse is also true. In practice the discrepancy is largest for features whose SHAP magnitudes lie close together in the middle of the ranking, since those are most sensitive to the weighting scheme. Top-1 or bottom-1 features rarely swap.

| Property | Histogram (fold-avg) | Beeswarm (pooled) |
|---|---|---|
| Fold weighting | Equal | Proportional to fold size |
| Shows distribution | No — one bar per feature | Yes — each dot is one sample |
| Shows direction (positive/negative) | No | Yes — blue = low feature value, red = high |
| Use case | Stable ranked list for feature selection | Understand how and in which direction each feature drives predictions |

In [1]:
sensor_shap_dfs[target]['feature'].head(20).tolist()

NameError: name 'sensor_shap_dfs' is not defined